In [65]:
sold = pd.read_csv("filtered_sold_residential.csv")
listings = pd.read_csv("filtered_listing_residential.csv")
import pandas as pd
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

/var/folders/gf/vxk2rst54wlb4z9gnqzsqs7m0000gn/T/ipykernel_5318/3801926522.py:1: DtypeWarning: Columns (0: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv("filtered_sold_residential.csv")
/var/folders/gf/vxk2rst54wlb4z9gnqzsqs7m0000gn/T/ipykernel_5318/3801926522.py:2: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("filtered_listing_residential.csv")


In [66]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
mortgage.groupby('year_month')[ 'rate_30yr_fixed']
.mean().reset_index())

In [67]:
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')
listings['year_month'] = pd.to_datetime(
listings['ListingContractDate']).dt.to_period('M')

In [68]:
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [69]:
# Check for any unmatched rows (rate should not be null)
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())
# Preview
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice',
'rate_30yr_fixed']].head())


297221
0
  CloseDate year_month  ClosePrice  rate_30yr_fixed
0       NaN        NaT         NaN              NaN
1       NaN        NaT         NaN              NaN
2       NaN        NaT         NaN              NaN
3       NaN        NaT         NaN              NaN
4       NaN        NaT         NaN              NaN


In [70]:
sold_with_rates.to_csv('sold_enriched.csv', index=False)
listings_with_rates.to_csv('listings_enriched.csv', index=False)